In [3]:
from huggingface_hub import login
login()

In [4]:
from huggingface_hub import whoami
whoami()

{'type': 'user',
 'id': '6abf5661edac9782827cd680',
 'name': 'akhilcv',
 'fullname': 'Akhil CV',
 'canPay': False,
 'billingMode': 'prepaid',
 'periodEnd': 1793491200,
 'isPro': False,
 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/p77J12XQB5TNrLQpRoaMW.png',
 'orgs': [],
 'auth': {'type': 'oauth', 'expiresAt': '2026-11-01T07:27:08.000Z'}}

In [1]:
import pandas as pd
from huggingface_hub import hf_hub_download
import duckdb

In [15]:
# daily 2025-26
nifty_2025_path = hf_hub_download(
    repo_id="rissin/nse-options-intraday",
    filename="historical_daily/NIFTY/NIFTY_2025.parquet",
    repo_type="dataset"
)
nifty_2026_path = hf_hub_download(
    repo_id="rissin/nse-options-intraday",
    filename="historical_daily/NIFTY/NIFTY_2026.parquet",
    repo_type="dataset"
)

# intraday 2026
file_path = hf_hub_download(
    repo_id="rissin/nse-options-intraday",
    filename="upstox_intraday/NIFTY/NIFTY_2026.parquet",
    repo_type="dataset"
)

c:\GIT\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\akhil\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


historical_daily/NIFTY/NIFTY_2025.parque(…): reconstructing file:   0%|          |  0.00B / 6.77MB            

historical_daily/NIFTY/NIFTY_2025.parque(…): downloading bytes:           |  0.00B            

c:\GIT\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\akhil\.cache\huggingface\hub\datasets--rissin--nse-options-intraday. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


historical_daily/NIFTY/NIFTY_2026.parque(…): reconstructing file:   0%|          |  0.00B / 4.03MB            

historical_daily/NIFTY/NIFTY_2026.parque(…): downloading bytes:           |  0.00B            

In [6]:
con = duckdb.connect()

query = """
SELECT
    date,
    timestamp,
    expiry,
    strike,
    option_type,
    open,
    high,
    low,
    close,
    volume,
    oi
FROM read_parquet(?)
WHERE date = '2026-03-30'
  AND strike = 21800
ORDER BY timestamp
"""

result = con.execute(query, [file_path]).df()
result.head()

,date,timestamp,expiry,strike,option_type,open,high,low,close,volume,oi
0,2026-03-30,2026-03-30 09:15:00+05:30,2026-03-30,21800.0,CE,799.15,799.15,673.40,673.40,1235,6500
1,2026-03-30,2026-03-30 09:15:00+05:30,2026-03-30,21800.0,PE,4.95,5.45,1.45,1.85,4819750,4502550
2,2026-03-30,2026-03-30 09:15:00+05:30,2026-04-07,21800.0,PE,181.00,233.45,181.00,225.50,35295,224965
3,2026-03-30,2026-03-30 09:15:00+05:30,2026-04-28,21800.0,PE,445.15,470.00,439.95,470.00,9100,68705
4,2026-03-30,2026-03-30 09:15:00+05:30,2026-04-13,21800.0,PE,300.00,340.50,289.75,336.85,4290,77350


In [17]:
query = """
SELECT
    date,
    timestamp,
    expiry,
    strike,
    option_type,
    open,
    high,
    low,
    close,
    volume,
    oi
FROM read_parquet(?)
WHERE date BETWEEN '2025-01-01' AND '2026-12-31'
  AND strike = 21800
ORDER BY date, expiry, strike, option_type
"""

result = con.execute(query, [[nifty_2025_path,nifty_2026_path]]).df()
result.head()

,date,timestamp,expiry,strike,option_type,open,high,low,close,volume,oi
0,2025-01-01,2025-01-01,2025-01-02,21800.0,CE,0.00,0.0,0.00,1776.65,0,0
1,2025-01-01,2025-01-01,2025-01-02,21800.0,PE,1.45,1.5,0.80,1.05,135791,1554450
2,2025-01-01,2025-01-01,2025-01-30,21800.0,CE,2067.60,2067.6,2067.60,2067.60,1,575
3,2025-01-01,2025-01-01,2025-01-30,21800.0,PE,18.75,20.4,12.75,13.80,22565,1238000
4,2025-01-02,2025-01-02,2025-01-02,21800.0,CE,0.00,0.0,0.00,1776.65,0,0


In [18]:
result.tail()


,date,timestamp,expiry,strike,option_type,open,high,low,close,volume,oi
2113,2026-06-29,2026-06-29,2026-07-14,21800.0,PE,4.1,4.90,4.1,4.90,47,3705
2114,2026-06-29,2026-06-29,2026-07-28,21800.0,CE,0.0,0.00,0.0,1822.55,0,130
2115,2026-06-29,2026-06-29,2026-07-28,21800.0,PE,17.3,17.95,13.7,15.35,354,121095
2116,2026-06-29,2026-06-29,2026-08-25,21800.0,CE,0.0,0.00,0.0,2537.50,0,0
2117,2026-06-29,2026-06-29,2026-08-25,21800.0,PE,33.1,35.65,30.4,33.75,197,31525


In [19]:
file_path = [nifty_2025_path,nifty_2026_path]
def get_options(file_path, s_date, e_date):
  query = """
    SELECT
      date,
      timestamp,
      expiry,
      strike,
      option_type,
      close
    FROM read_parquet(?)
    WHERE date BETWEEN ? AND ?
    """
  result = duckdb.execute( query,
      [file_path, s_date, e_date]).df()
  return result

In [43]:
niftyoptions_df = get_options(file_path, '2025-01-01', '2026-12-31' )
niftyspot_df = pd.read_csv('nifty_close.csv')

In [66]:
niftyoptions_df.head(1)

,date,timestamp,expiry,strike,option_type,close,date_dt,exdate_dt,time_to_expiry,days_to_exp,ttm
0,2025-01-01,2025-01-01,2025-01-30,26150.0,CE,8.1,2025-01-01,2025-01-30,29 days,29,0.079452


In [69]:
import numpy as np

def optiondata_transform(niftyoptions_df, niftyspot_df):

    niftyoptions_df['date_dt'] = pd.to_datetime(niftyoptions_df['date'], format = '%Y-%m-%d')
    niftyoptions_df['exdate_dt'] = pd.to_datetime(niftyoptions_df['expiry'], format = '%Y-%m-%d')
    niftyoptions_df['time_to_expiry'] = niftyoptions_df['exdate_dt'] - niftyoptions_df['date_dt']
    niftyoptions_df['days_to_exp'] = niftyoptions_df['time_to_expiry'].dt.days
    #calendar dates difference, use 356 days for time to maturity
    niftyoptions_df['ttm'] = niftyoptions_df['days_to_exp'] / 365
    niftyspot_df['date_dt'] = pd.to_datetime(niftyspot_df['date'], format = "%Y-%m-%d")
    observation_dates = niftyoptions_df['date_dt'].unique()
    spot = np.zeros(len(observation_dates))
    for i in range(len(observation_dates)):
        spot[i] = niftyspot_df[niftyspot_df['date_dt'] == observation_dates[i]]['close'].item()
    return niftyoptions_df, niftyspot_df, spot, observation_dates

def data_day(date, niftyoptions_df, niftyspot_df):
    """
    take the pre-processed niftyoptions_df (previous function) and niftyspot_df
    return the dataframe with OTM calls and puts and estimated risk-free
    interest rate r
    """
    df_temp = niftyoptions_df[niftyoptions_df['date_dt'] == date].reset_index()
    df_temp['spot'] = niftyspot_df[niftyspot_df['date_dt'] == date]['close'].item()
    df_temp['moneyness'] = df_temp['strike'] / (df_temp['spot'])
    # df_temp['mid_price'] = 0.5 * (df_temp['best_offer'] + df_temp['best_bid'])
    df_temp['mid_price'] = df_temp['close']

    #pair up calls and puts to calculate the implied risk-free interest rate
    call_df = df_temp[df_temp['option_type'] == 'CE']
    put_df = df_temp[df_temp['option_type'] == 'PE']
    merged_df = pd.merge(call_df, put_df, on=['strike', 'expiry', 'spot', 'ttm'])
    #put-call parity
    r_temp = np.log((merged_df['strike'] )/ (merged_df['spot'] - merged_df['mid_price_x'] + merged_df['mid_price_y'])) / merged_df['ttm']
    df_temp['r'] = np.median(r_temp)
    r = np.median(r_temp)
    #out-of-the-money calls and puts
    subset_df = df_temp[((df_temp['option_type'] == 'CE') & (df_temp['moneyness'] >= 1)) | ((df_temp['option_type'] == 'PE') & (df_temp['moneyness'] < 1))]
    return subset_df, r

In [52]:
niftyoptions_df, niftyspot_df, spot, observation_dates = optiondata_transform(niftyoptions_df, niftyspot_df)

In [62]:
niftyoptions_df.head(1)

,date,timestamp,expiry,strike,option_type,close,date_dt,exdate_dt,time_to_expiry,days_to_exp,ttm
0,2025-01-01,2025-01-01,2025-01-30,26150.0,CE,8.1,2025-01-01,2025-01-30,29 days,29,0.079452


In [71]:
subset_df, r = data_day(observation_dates[0], niftyoptions_df, niftyspot_df)
subset_df

,index,date,timestamp,expiry,strike,option_type,close,date_dt,exdate_dt,time_to_expiry,days_to_exp,ttm,spot,moneyness,mid_price,r
0,0,2025-01-01,2025-01-01,2025-01-30,26150.0,CE,8.10,2025-01-01,2025-01-30,29 days,29,0.079452,23742.9,1.101382,8.10,0.099836
1,1,2025-01-01,2025-01-01,2025-01-02,22400.0,PE,1.70,2025-01-01,2025-01-02,1 days,1,0.002740,23742.9,0.943440,1.70,0.099836
3,3,2025-01-01,2025-01-01,2025-02-27,22400.0,PE,93.20,2025-01-01,2025-02-27,57 days,57,0.156164,23742.9,0.943440,93.20,0.099836
4,4,2025-01-01,2025-01-01,2025-01-09,22400.0,PE,4.90,2025-01-01,2025-01-09,8 days,8,0.021918,23742.9,0.943440,4.90,0.099836
5,5,2025-01-01,2025-01-01,2025-01-02,25150.0,CE,0.80,2025-01-01,2025-01-02,1 days,1,0.002740,23742.9,1.059264,0.80,0.099836
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1555,1555,2025-01-01,2025-01-01,2025-01-16,24950.0,CE,13.55,2025-01-01,2025-01-16,15 days,15,0.041096,23742.9,1.050840,13.55,0.099836
1557,1557,2025-01-01,2025-01-01,2028-06-29,20000.0,PE,400.00,2025-01-01,2028-06-29,1275 days,1275,3.493151,23742.9,0.842357,400.00,0.099836
1559,1559,2025-01-01,2025-01-01,2029-12-27,20000.0,PE,202.30,2025-01-01,2029-12-27,1821 days,1821,4.989041,23742.9,0.842357,202.30,0.099836
1560,1560,2025-01-01,2025-01-01,2025-02-27,24950.0,CE,161.90,2025-01-01,2025-02-27,57 days,57,0.156164,23742.9,1.050840,161.90,0.099836
